In [1]:
#Import libraries
import numpy as np
import pandas as pd

In [7]:
# Read the CSV file
data = pd.read_csv('Lifespan/companyDrug/speeds_and_coordinates_20241016_3_updated.csv')

In [8]:
# Display the first few rows of the dataset
print(data.head())
#Summary of data
print(data.describe())

   Frame       Speed           X           Y  Changed Pixels
0      1   14.152425  476.356932  487.840708             460
1      2   21.822664  419.305000  511.537500             494
2      3   22.484239  430.031250  509.536318             200
3      4  120.411300  453.283636  505.876364             310
4      5  103.131087  414.000000  551.500000             200
              Frame         Speed             X             Y  Changed Pixels
count  75593.000000  75593.000000  74855.000000  74855.000000    75593.000000
mean    5400.000000      1.420182    347.420578    497.622615       23.145662
std     3117.423385     11.178094    121.705676    129.644571       42.538445
min        1.000000      0.000000    124.005291    159.162088        0.000000
25%     2700.000000      0.000000    249.765571    390.323443        0.000000
50%     5400.000000      0.275513    336.299479    513.923404        8.000000
75%     8100.000000      1.076040    425.747680    632.201238       30.000000
max    107

In [9]:
#Add Euclidean Speed to the next column
data['euclidean_speed'] = np.sqrt((data['X'].diff())**2 + (data['Y'].diff())**2) * 2
data['euclidean_speed'] = data['euclidean_speed'].shift(-1)

In [17]:
print(data[10790:10810])

       Frame     Speed           X           Y  Changed Pixels  \
10790  10791  1.357185  483.314961  326.992126              21   
10791  10792  0.815473  483.754545  327.509091              19   
10792  10793  0.800227  483.800000  327.914286              16   
10793  10794  3.378011  483.552381  328.228571              28   
10794  10795  7.026053  481.868217  328.100775              64   
10795  10796  4.478708  478.355191  328.098361              43   
10796  10797  3.753930  476.116505  328.043689              33   
10797  10798  1.051998  474.266968  327.723982              26   
10798  10799  1.658044  474.027397  327.255708              43   
10799      1  4.779959  358.559406  426.381188              99   
10800      2  3.475262  358.983051  424.029056              71   
10801      3  0.942863  359.553571  422.387755              24   
10802      4  0.160443  359.514925  421.917910               2   
10803      5  0.444741  359.524876  421.997512              18   
10804     

In [6]:
# Step 1: Check for "no significant movement" with a tolerance of 0.5
tolerance = 0.5
data['no_movement'] = (data['X'].diff().fillna(0).abs() < tolerance) & (data['Y'].diff().fillna(0).abs() < tolerance)

# Step 2: Count consecutive no-movement frames
# Use a cumulative counter that resets when movement is detected
data['no_movement_count'] = data['no_movement'].cumsum() - data['no_movement'].cumsum().where(~data['no_movement']).ffill().fillna(0).astype(int)

# Step 3: Identify when no movement reaches the threshold
threshold = 100
data['dead'] = data['no_movement_count'] >= threshold

# Step 4: Record the first row where the worm is declared dead
dead_row = data[data['dead']].index.min()

if pd.notna(dead_row):
    print(f"The worm is declared dead at row index: {dead_row}")
    # Display relevant columns for verification
    print(data[['Frame', 'X', 'Y', 'no_movement', 'no_movement_count', 'dead']].tail(15))
else:
    print("The worm never reaches the dead threshold.")



The worm is declared dead at row index: 4761
       Frame           X           Y  no_movement  no_movement_count   dead
75578  10785  252.257426  413.062706         True                 32  False
75579  10786  252.257426  413.062706         True                 33  False
75580  10787  252.305921  413.003289         True                 34  False
75581  10788  252.085246  413.131148         True                 35  False
75582  10789  252.191223  413.514107         True                 36  False
75583  10790  252.191223  413.514107         True                 37  False
75584  10791  252.191223  413.514107         True                 38  False
75585  10792  252.201258  413.493711         True                 39  False
75586  10793  252.201258  413.493711         True                 40  False
75587  10794  252.201258  413.493711         True                 41  False
75588  10795  252.201258  413.493711         True                 42  False
75589  10796  252.094937  413.534810       